#### Supplemental Figure 2 C, D theta phase

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.pyplot import cm
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib as mpl
from paths import DATA_DIR, fig_dir



In [ ]:
from spyglass.common import *
from spyglass.lfp import *
from spyglass.lfp.v1 import *
from spyglass.lfp.analysis.v1 import *

In [ ]:
from plot_figs_summary_metrics import get_all_rat_big_dfs, get_all_rat_stable_nwb_file_names
from fig_helpers import *
from find_my_data import *
from alison_decoding import ClusterlessAcausalResultsSummary


In [ ]:
set_figure_defaults()

fig_path = fig_dir('figs26')
if not os.path.exists(fig_path):
    os.makedirs(fig_path)

save_fig = False

subject_ids = ['senor', 'chimi', 'j16', 'wilbur', 'peanut', 'allrats']
custom_colors_by_rat = iter(cm.tab20b([0,.8, .85, .1, .05]))


### set up to plot theta

In [ ]:
### functions

def bin_df(concatenated_df, bins):
    #concatenated_df['bin'] = pd.cut(concatenated_df['electrode_cc'], bins=bins)
    concatenated_df['bin'] = pd.cut(concatenated_df['electrode_cc'], bins=bins)
    
    # Count Occurrences and Calculate Density and Normalize
    binned_data = concatenated_df.groupby(['nwb_file_name', 'epoch_interval_list_name', 'bin']).size().reset_index(name='Count')
    total_counts = binned_data.groupby(['nwb_file_name', 'epoch_interval_list_name'])['Count'].sum().reset_index(name='Total_Count')
    binned_data = pd.merge(binned_data, total_counts, on=['nwb_file_name', 'epoch_interval_list_name'])
    binned_data['Density'] = binned_data['Count'] / binned_data['Total_Count']
    bin_width = (bins[-1]-bins[0])/(len(bins)-1)
    binned_data['Density_norm'] = binned_data['Density']/bin_width

    #binned_data['subject_id'] = binned_data['subject_id'].astype('str')
    binned_data['nwb_file_name'] = binned_data['nwb_file_name'].astype('str')
    binned_data['epoch_interval_list_name'] = binned_data['epoch_interval_list_name'].astype('str')
    binned_data['bin'] = binned_data['bin'].astype('str')

    return binned_data


def bin_theta_data_density_norm(binned_theta_data_density_norm_dict, concatenated_dfs_by_subject, bins, subject_ids):
    for subject_id in subject_ids:
        binned_theta_data_density_norm_dict[subject_id] = {}
        binned_theta_data_density_norm_dict[subject_id]['first'] = {}
        binned_theta_data_density_norm_dict[subject_id]['last'] = {}
        for seg in ['first','last']:
            binned_theta_data_density_norm_dict[subject_id][seg]['stay'] = {}
            binned_theta_data_density_norm_dict[subject_id][seg]['switch'] = {}
            binned_theta_data_density_norm_dict[subject_id][seg]['both'] = {}
        
        big_theta = concatenated_dfs_by_subject[subject_id]
        
        # identify First and Last Segment data
        big_theta_first = big_theta[big_theta['is_first_seg_of_trial']==True]
        big_theta_last = big_theta[big_theta['is_last_seg_of_trial']==True]
        
        for i,big_theta_data in enumerate([big_theta_first, big_theta_last]):
            for trial_type in ['stay', 'switch', 'both']:
                if trial_type == 'stay':
                    big_theta_data_trial_type = big_theta_data[big_theta_data['stem_switch']==False]
                elif trial_type == 'switch':
                    big_theta_data_trial_type = big_theta_data[big_theta_data['stem_switch']==True]
                elif trial_type == 'both':
                    big_theta_data_trial_type = big_theta_data
            
                # FILTER IN MANY WAYS    

                #summary for all trial types, then again just for stay trials, and again for just switch trials
    #             overall phase of local reps
    #             overall phase of nonlocal by seg
    #             overall phase of nonlocal by patch
                big_theta_data_local = big_theta_data_trial_type[big_theta_data_trial_type['nonlocal_by_segment']==False]
                big_theta_data_nonlocal_by_seg = big_theta_data_trial_type[big_theta_data_trial_type['nonlocal_by_segment']==True]
                big_theta_data_nonlocal_by_patch = big_theta_data_trial_type[big_theta_data_trial_type['nonlocal_by_patch']==True]
                big_theta_data_nonlocal_by_seg_in_patch = big_theta_data_nonlocal_by_seg[big_theta_data_nonlocal_by_seg['nonlocal_by_patch']==False]

    #             stay
    #             switch
    #             # and also make the versions labelled 
    #             chosen
    #             alternative
                big_theta_data_nonlocal_by_seg_stayrep = big_theta_data_nonlocal_by_seg[np.logical_and(
                    big_theta_data_nonlocal_by_seg['is_mental_seg_mapped_a_leaf']==True,
                    big_theta_data_nonlocal_by_seg['nonlocal_by_patch']==False
                )]
                big_theta_data_nonlocal_by_seg_switchrep = big_theta_data_nonlocal_by_seg[~np.logical_and(
                    big_theta_data_nonlocal_by_seg['is_mental_seg_mapped_a_leaf']==True,
                    big_theta_data_nonlocal_by_seg['nonlocal_by_patch']==False
                )]
                big_theta_data_nonlocal_by_seg_chosenrep = big_theta_data_nonlocal_by_seg[np.logical_or(
                    np.logical_and(big_theta_data_nonlocal_by_seg['stem_switch']==True,
                                  ~np.logical_and(big_theta_data_nonlocal_by_seg['is_mental_seg_mapped_a_leaf']==True,
                                                big_theta_data_nonlocal_by_seg['nonlocal_by_patch']==False)),
                    np.logical_and(big_theta_data_nonlocal_by_seg['stem_switch']==False,
                                   np.logical_and(big_theta_data_nonlocal_by_seg['is_mental_seg_mapped_a_leaf']==True,
                                    big_theta_data_nonlocal_by_seg['nonlocal_by_patch']==False))
                )]
                big_theta_data_nonlocal_by_seg_altrep = big_theta_data_nonlocal_by_seg[np.logical_or(
                    np.logical_and(big_theta_data_nonlocal_by_seg['stem_switch']==True,
                                  np.logical_and(big_theta_data_nonlocal_by_seg['is_mental_seg_mapped_a_leaf']==True,
                                                big_theta_data_nonlocal_by_seg['nonlocal_by_patch']==False)),
                    np.logical_and(big_theta_data_nonlocal_by_seg['stem_switch']==False,
                                   ~np.logical_and(big_theta_data_nonlocal_by_seg['is_mental_seg_mapped_a_leaf']==True,
                                    big_theta_data_nonlocal_by_seg['nonlocal_by_patch']==False))
                )]

    #             in_patch_stay
    #             in_patch_switch
    #             # and also make the versions labelled 
    #             in_patch_chosen
    #             in_patch_alterantive
                big_theta_data_nonlocal_by_seg_in_patch_stayrep = big_theta_data_nonlocal_by_seg_in_patch[
                    big_theta_data_nonlocal_by_seg_in_patch['is_mental_seg_mapped_a_leaf']==True]
                big_theta_data_nonlocal_by_seg_in_patch_switchrep = big_theta_data_nonlocal_by_seg_in_patch[
                    big_theta_data_nonlocal_by_seg_in_patch['is_mental_seg_mapped_a_leaf']==False]
                big_theta_data_nonlocal_by_seg_in_patch_chosenrep = big_theta_data_nonlocal_by_seg_in_patch[np.logical_or(
                    np.logical_and(big_theta_data_nonlocal_by_seg_in_patch['stem_switch']==True,
                                   big_theta_data_nonlocal_by_seg_in_patch['is_mental_seg_mapped_a_leaf']==False),
                    np.logical_and(big_theta_data_nonlocal_by_seg_in_patch['stem_switch']==False,
                                   big_theta_data_nonlocal_by_seg_in_patch['is_mental_seg_mapped_a_leaf']==True),
                    )]
                big_theta_data_nonlocal_by_seg_in_patch_altrep = big_theta_data_nonlocal_by_seg_in_patch[np.logical_or(
                    np.logical_and(big_theta_data_nonlocal_by_seg_in_patch['stem_switch']==True,
                                   big_theta_data_nonlocal_by_seg_in_patch['is_mental_seg_mapped_a_leaf']==True),
                    np.logical_and(big_theta_data_nonlocal_by_seg_in_patch['stem_switch']==False,
                                   big_theta_data_nonlocal_by_seg_in_patch['is_mental_seg_mapped_a_leaf']==False),
                    )]

                # BIN THOSE FILTERED DATA all of them and give them all their own names with the appropriate FIRST or FINAL subset as well
                if i==0:
                    seg = 'first'
                elif i==1:
                    seg = 'last'
                binned_theta_data_density_norm_dict[subject_id][seg][trial_type]['local'] = bin_df(big_theta_data_local, bins)
                binned_theta_data_density_norm_dict[subject_id][seg][trial_type]['nonlocal_by_seg'] = bin_df(big_theta_data_nonlocal_by_seg, bins)
                binned_theta_data_density_norm_dict[subject_id][seg][trial_type]['nonlocal_by_patch'] = bin_df(big_theta_data_nonlocal_by_patch, bins)
                binned_theta_data_density_norm_dict[subject_id][seg][trial_type]['nonlocal_by_seg_in_patch'] = bin_df(big_theta_data_nonlocal_by_seg_in_patch, bins)
                binned_theta_data_density_norm_dict[subject_id][seg][trial_type]['nonlocal_by_seg_stayrep'] = bin_df(big_theta_data_nonlocal_by_seg_stayrep, bins)
                binned_theta_data_density_norm_dict[subject_id][seg][trial_type]['nonlocal_by_seg_switchrep'] = bin_df(big_theta_data_nonlocal_by_seg_switchrep, bins)
                binned_theta_data_density_norm_dict[subject_id][seg][trial_type]['nonlocal_by_seg_chosenrep'] = bin_df(big_theta_data_nonlocal_by_seg_chosenrep, bins)
                binned_theta_data_density_norm_dict[subject_id][seg][trial_type]['nonlocal_by_seg_altrep'] = bin_df(big_theta_data_nonlocal_by_seg_altrep, bins)
                binned_theta_data_density_norm_dict[subject_id][seg][trial_type]['nonlocal_by_seg_in_patch_stayrep'] = bin_df(big_theta_data_nonlocal_by_seg_in_patch_stayrep, bins)
                binned_theta_data_density_norm_dict[subject_id][seg][trial_type]['nonlocal_by_seg_in_patch_switchrep'] = bin_df(big_theta_data_nonlocal_by_seg_in_patch_switchrep, bins)
                binned_theta_data_density_norm_dict[subject_id][seg][trial_type]['nonlocal_by_seg_in_patch_chosenrep'] = bin_df(big_theta_data_nonlocal_by_seg_in_patch_chosenrep, bins)
                binned_theta_data_density_norm_dict[subject_id][seg][trial_type]['nonlocal_by_seg_in_patch_altrep'] = bin_df(big_theta_data_nonlocal_by_seg_in_patch_altrep, bins)                
                
    return binned_theta_data_density_norm_dict



def plot_theta_comparison_fmt(binned_theta_data_density_norm_dict, subject_ids_all, segs, trial_types, data_groups, linestyles,
                          err_style, ci, fig_path, save_fig, figwidth, figheight):
    for seg in segs:
        for trial_type in trial_types:
            subject_ids = subject_ids_all.copy()
            fig, ax = plt.subplots(figsize=(figwidth,figheight))
            for i,data_group in enumerate(data_groups):
                colors = iter(cm.tab20b([0,.8, .85, .1, .05]))
                for subject_id in subject_ids:
                    df = binned_theta_data_density_norm_dict[subject_id][seg][trial_type][data_group]
                    sns.lineplot(data=df, x='bin_shift_early_late', y='Density_norm',
                             color=next(colors) if subject_id != 'allrats' else 'black',
                             linestyle=linestyles[i],
                             linewidth=1 if subject_id !='allrats' else 2,
                                 alpha=.3 if subject_id != 'allrats' else 1,
                             label = f'_Rat {subject_id[0].upper()}, {data_group}',
                                 err_style=err_style,
                                 markers=True,
                                ci=ci if subject_id=='allrats' else 0)
            plt.xlabel("Theta phase")
    #         plt.xticks([x for x in plt.xticks()[0]], labels=[str(round(b, 2)) for b in bins], rotation=45)
            plt.xticks(np.arange(0,13,3)-0.5, labels=[str(round(b, 2)) for b in bins][::3], rotation=0)
            plt.axvline(x=5.5, linestyle=':', c='lightgrey')
            plt.ylim(0, 0.35)#plt.ylim()[1]+.05)
            plt.ylabel("Density")
            plt.text(0.05,0.95,'Early',color='lightgrey',va='top', ha='left', transform=plt.gca().transAxes)
            plt.text(0.95,0.95,'Late',color='lightgrey', va='top', ha='right', transform=plt.gca().transAxes)
            plt.title(f"Segment: {seg}, Trial_type: {trial_type},\n{data_groups[0]} vs {data_groups[1]}", fontsize=6)
#             plt.legend(bbox_to_anchor=(1.05,1), loc='upper left', fontsize=6, frameon=False)
            plt.xticks(rotation=0)
            sns.despine(offset=5)
            if save_fig:
                fig_name = f'new_subj{len(subject_ids)}_thetaphase_seg{seg}_trial{trial_type}_{data_groups[0]}_{data_groups[1]}_errstyle{err_style}_ci{ci}_w{figwidth}_h{figheight}'
                plt.savefig(f'{fig_path}{fig_name}.pdf', format='pdf', bbox_inches="tight", pad_inches=.5)
            plt.show()

### load data

In [ ]:
# Params
behavior_model_params_name = 'default_hmm_0623' #'hmm_test' #'default_hmm'

position_info_param_name='default_decoding'
remove_hpd_timepoints = True
hpd_percent = 50
hpd_threshold = 50
require_nonlocal_by_segment = False
remove_low_speed_timepoints = True
head_speed_threshold = 10

out_path = f'{DATA_DIR}/big_df_pkls/'
# today_now = datetime.now().strftime("%Y%m%d") 
today_now = '20240212'
subject_ids = ['senor', 'chimi', 'j16', 'wilbur', 'peanut']

big_dfs = {}
for subject_id in subject_ids:
    try:
#         if subject_id == 'senor':
#             senor_big_df = pd.read_pickle(out_path+subject_id.lower()+'_big_df_RL_deltaq_stable'+today_now+'.pkl')
#         if subject_id == 'chimi':
#             chimi_big_df = pd.read_pickle(out_path+subject_id+'_big_df_RL_deltaq_stable'+today_now+'.pkl')
#         if subject_id == 'wilbur':
#             wilbur_big_df = pd.read_pickle(out_path+subject_id+'_big_df_RL_deltaq_stable'+today_now+'.pkl')
#         if subject_id == 'peanut':
#             peanut_big_df = pd.read_pickle(out_path+subject_id+'_big_df_RL_deltaq_stable'+today_now+'.pkl')
#         if subject_id == 'j16':
#             j16_big_df = pd.read_pickle(out_path+subject_id+'_big_df_RL_uncertainty_'+today_now+'.pkl')
        big_dfs[subject_id] = pd.read_pickle(out_path+subject_id.lower()+'_big_df_RL_deltaq_stable'+today_now+'.pkl')
    except Exception as e:
        print('exception',e)

stable_nwbs = {}
clusterless_nwbs = {}
stable_clusterless_nwbs = {}
for subject_id in subject_ids:
    stable_nwbs[subject_id] = list( (Session & {'session_description LIKE "Spatial bandit task (regular)"'}
                                             & {"subject_id": subject_id}).fetch('nwb_file_name') )
    clusterless_nwbs[subject_id] = list(np.unique((ClusterlessAcausalResultsSummary()
                                                   & spatial_bandit_query_by_rat(rat_list=[subject_id])).fetch('nwb_file_name')))
    if subject_id == 'j16':
        stable_nwbs['j16'].remove('mediumnwb20230802_.nwb')
    if subject_id == 'chimi':
        stable_nwbs['chimi'].remove('chimi20200216_new_.nwb')
    if subject_id == 'senor':
        stable_nwbs['senor'].remove('senor20201030_.nwb')

    stable_clusterless_nwbs[subject_id] = [nwb for nwb in clusterless_nwbs[subject_id] if nwb in stable_nwbs[subject_id]]

print(stable_clusterless_nwbs)

is_mapped_seg_a_leaf_map = {0:False, 1:True, 2:True, 3:False, 4:True, 5:True, 6:False, 7:True, 8:True}
segs_to_patch_map = {0:1, 1:1, 2:1, 3:2, 4:2, 5:2, 6:3, 7:3, 8:3}

# get to stable data only
all_rat_big_dfs_stable = {}
for subject_id in subject_ids:
    df = big_dfs[subject_id]
    df_stable = df[df['nwb_file_name'].isin(stable_clusterless_nwbs[subject_id])]
    df_stable['is_actual_seg_mapped_a_leaf'] = df_stable[['actual_segment_mapped']].applymap(is_mapped_seg_a_leaf_map.get)
    df_stable['is_mental_seg_mapped_a_leaf'] = df_stable[['mental_segment_mapped']].applymap(is_mapped_seg_a_leaf_map.get)
    df_stable['mental_patch_mapped'] = df_stable[['mental_segment_mapped']].applymap(segs_to_patch_map.get)
    all_rat_big_dfs_stable[subject_id] = df_stable

for subject_id in subject_ids:
    df = all_rat_big_dfs_stable[subject_id]
    p_rew_cols = [f"p_rew_leaf{i}" for i in [1,2,3,4,5,6]]
    all_rat_big_dfs_stable[subject_id] = all_rat_big_dfs_stable[subject_id][~all_rat_big_dfs_stable[subject_id][p_rew_cols].eq(all_rat_big_dfs_stable[subject_id]['p_rew_leaf1'], axis=0).all(axis=1)]

In [ ]:

all_nwb_file_names_dict, stable_nwb_file_names_dict = get_all_rat_stable_nwb_file_names(all_rat_big_dfs_stable)

In [ ]:

# LFP PARAMS
filter_name = "theta_5_11"
use_first_available_elect_id = True

# Allocate
bigdf_lfpband_byrat_byepoch = {}

for i, (subject_id, big_df) in enumerate(all_rat_big_dfs_stable.items()): #subject_id = 'j16'
    # ^Loads big_df for the rat and rat name
    # These do not all need to be fstrings but it is ok
    bigdf_lfpband_byrat_byepoch[f'{subject_id}'] = {}
    
    nwb_file_names = stable_nwb_file_names_dict[f'{subject_id}']
    
    for nwb_file_name in nwb_file_names:
        # Iterate through the run epochs of data within one nwb day of data
        #nwb_file_name = "j1620210710_.nwb"
        bigdf_lfpband_byrat_byepoch[f'{subject_id}'][f'{nwb_file_name}'] = {}
        # Get run epochs during this day that have LFPband data
        epoch_names = np.unique((LFPBandV1
                                 & {"nwb_file_name":nwb_file_name, "filter_name":filter_name}
                                 & {'target_interval_list_name LIKE "%noPrePostTrialTimes"'}
                                 ).fetch('target_interval_list_name'))
        for epoch_name in epoch_names:
            bigdf_lfpband_byrat_byepoch[f'{subject_id}'][f'{nwb_file_name}'][f'{epoch_name}'] = {}
            epoch_interval_list_name = epoch_name[:5]
            df_epoch = big_df[(big_df['nwb_file_name']==nwb_file_name)
                              & (big_df['epoch_interval_list_name']==epoch_interval_list_name)
                              & (big_df["is_nosepoking"]==False)]
            lfp_band_table = LFPBandV1 & {"nwb_file_name":nwb_file_name, "filter_name":filter_name, "target_interval_list_name":epoch_name}
            lfp_df = lfp_band_table.fetch1_dataframe()
            lfp_band_eseries = lfp_band_table.fetch_nwb()[0][
                "lfp_band"
            ]
            lfp_band_electrode_ids = lfp_band_eseries.electrodes[:].index.values
            lfp_band_elect_indices = get_electrode_indices(lfp_band_eseries, lfp_band_electrode_ids)
            lfp_band_timestamps = np.asarray(lfp_band_eseries.timestamps)

            # phase_df = lfp_band_table.compute_signal_phase(electrode_list = lfp_band_electrode_ids) # update for truth val of array is ambiguous in lfp band sg code.
            phase_df = lfp_band_table.compute_signal_phase(electrode_list = list(lfp_band_electrode_ids))
            # if want to also keep power in future
            # power_df = lfp_band_table.compute_signal_power(electrode_list = lfp_band_electrode_ids)
            # assert phase_df.index.equals(power_df.index), "Indices of phase and power dataframes are not equal, check that they have the same timestamps before merging"
            # merged_bigdf_lfpband = pd.merge(df_epoch, merged_lfp_data, left_index=True, right_index=True, how='outer')
            merged_bigdf_lfpband = pd.merge(df_epoch, phase_df, left_index=True, right_index=True, how='outer')
            
            if use_first_available_elect_id:
                electrode = lfp_band_electrode_ids[0]
                interp_lfpband_at_decode = merged_bigdf_lfpband[[f"electrode {electrode}"]].interpolate(method='index')
                merged_bigdf_lfpband[f"electrode {electrode}"] = interp_lfpband_at_decode
                merged_bigdf_lfpband.rename(columns={f'electrode {electrode}': 'electrode_cc'}, inplace=True) # so that the column name is the same across rats!
                bigdf_lfpband = merged_bigdf_lfpband[merged_bigdf_lfpband["nwb_file_name"].notna()] # only use the indices from the original big_df
            else:
                raise NotImplementedError("Need to implement interpolation of LFP band data at decode time for each elect id of interest")

            # Add epoch data to bigdf_lfpband_byrat_byepoch
            bigdf_lfpband_byrat_byepoch[f'{subject_id}'][f'{nwb_file_name}'][f'{epoch_name}'] = bigdf_lfpband

In [ ]:

concatenated_dfs_by_subject = {}

for subject_id in bigdf_lfpband_byrat_byepoch.keys():
    keys = []
    dfs = []
    for nwb_file_name, epochs in bigdf_lfpband_byrat_byepoch[subject_id].items():
        for epoch, df in epochs.items():
            keys.append((nwb_file_name, epoch))
            dfs.append(df)
            
    concatenated_df = pd.concat(dfs, keys=keys, names=['nwb_file_name', 'lfpband_epoch', 'time'])
    concatenated_df.reset_index(level=['nwb_file_name', 'lfpband_epoch'], drop=True, inplace=True)
    full_flat_df = concatenated_df.reset_index()
    concatenated_dfs_by_subject[subject_id] = full_flat_df

In [ ]:
all_rat_data = pd.concat(concatenated_dfs_by_subject.values(), keys=concatenated_dfs_by_subject.keys(), names=['subject_id'])
all_rat_data.reset_index(level=0, inplace=True)
all_rat_data.reset_index(drop=False, inplace=True)
concatenated_dfs_by_subject['allrats'] = all_rat_data

In [ ]:
bins = np.arange(0,2*np.pi+np.pi/12, np.pi/6)

binned_theta_data_density_norm_dict = {}
binned_theta_data_density_norm_dict = bin_theta_data_density_norm(binned_theta_data_density_norm_dict, concatenated_dfs_by_subject, bins, subject_ids)

### plot

In [ ]:
seg = 'first'
trial_type = 'stay'

bin_names = np.unique(binned_theta_data_density_norm_dict[subject_id][seg][trial_type]['nonlocal_by_seg'].bin)

mid = len(bin_names)//2
late_phase_bin_names = bin_names[0:mid]
early_phase_bin_names = bin_names[mid:]

bin_names_reordered = np.concatenate((early_phase_bin_names,late_phase_bin_names))
bin_mapping = {old:new for old,new in zip(bin_names, bin_names_reordered)}

segs = ['first','last']
trial_types = ['stay','switch','both']
data_groups = ['local', 'nonlocal_by_seg']
# ,'nonlocal_by_patch', 'nonlocal_by_seg_in_patch',
#                    'nonlocal_by_seg_stayrep','nonlocal_by_seg_switchrep',
#                    'nonlocal_by_seg_chosenrep','nonlocal_by_seg_altrep',
#                    'nonlocal_by_seg_in_patch_stayrep','nonlocal_by_seg_in_patch_switchrep',
#                    'nonlocal_by_seg_in_patch_chosenrep','nonlocal_by_seg_in_patch_altrep',
#                   ]

for seg in segs:
    for trial_type in trial_types:
        for data_group in data_groups:
            for subject_id in subject_ids:
                binned_theta_data_density_norm_dict[subject_id][seg][trial_type][data_group]['bin_shift_early_late'] = binned_theta_data_density_norm_dict[subject_id][seg][trial_type][data_group]['bin'].map(bin_mapping)
                binned_theta_data_density_norm_dict[subject_id][seg][trial_type][data_group]['bin_shift_early_late'] = pd.Categorical(
                     binned_theta_data_density_norm_dict[subject_id][seg][trial_type][data_group]['bin_shift_early_late'],
                     categories=bin_names,
                     ordered=True)

In [ ]:
segs = ['first','last']
trial_types = ['stay','switch']

linestyles = ['--','-']

err_style = 'band' #'bars'
ci=95

figwidth = 1.7 #TWO_COLUMN
figheight = 1.7 #TWO_COLUMN

In [ ]:

data_groups_all = [['local', 'nonlocal_by_seg']]
        
for data_groups in data_groups_all:
    plot_theta_comparison_fmt(binned_theta_data_density_norm_dict, subject_ids, segs, trial_types, data_groups, linestyles,
                          err_style, ci, fig_path, save_fig, figwidth, figheight)

In [ ]:
# stats
pvals = {}          

segs = ['is_first_seg_of_trial','is_last_seg_of_trial']
is_switch = [True,False]

for subject_id in subject_ids:
    subject_df = concatenated_dfs_by_subject[subject_id]
    print(subject_id, ': ', len(subject_df))
    for switch in is_switch:
        subject_trial_df = subject_df[subject_df['stem_switch'] == switch]
        print(switch, ' is_switch: ', len(subject_trial_df))
        for seg in segs:
            subject_trial_seg_df = subject_trial_df[subject_trial_df[seg]==True]
            print(seg, 'true: ', len(subject_trial_seg_df))
            
            subject_trial_seg_df_local = subject_trial_seg_df[subject_trial_seg_df.nonlocal_by_segment == False]
            subject_trial_seg_df_nonlocal = subject_trial_seg_df[subject_trial_seg_df.nonlocal_by_segment == True]
            print('local: ', len(subject_trial_seg_df_local), '\nnonlocal: ', len(subject_trial_seg_df_nonlocal))
            
            phase_local = subject_trial_seg_df_local.electrode_cc.values
            phase_nonlocal = subject_trial_seg_df_nonlocal.electrode_cc.values
            
            print('local nan len: ',sum(np.isnan(phase_local)))
            print('nonlocal nan len: ', sum(np.isnan(phase_nonlocal)))
            
            # # save out local nonlocal data
            file_path = f'{DATA_DIR}/figs26/'
            file_name = f'theta_{subject_id}_switch{switch}_seg{seg}True_electrode_cc_values_phase_data_'
            
            np.save(file_path+file_name+'local', phase_local)
            np.save(file_path+file_name+'nonlocal',phase_nonlocal)
            
            # stats are in separate nb for separate env if needed